In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

In [ ]:
def draw_and_compare(true_slope, true_int, sample_size):
    x = np.random.normal(50, 5, sample_size)
    xlims = np.array([np.min(x), np.max(x)])
    errors = np.random.normal(0, 6, sample_size)
    y = (true_slope * x + true_int) + errors
    sample = Table().with_columns('x', x, 'y', y)

    sample.scatter('x', 'y')
    plots.plot(xlims, true_slope*xlims + true_int, lw=2, color='green')
    plots.title('True Line, and Points Created')

    sample.scatter('x', 'y')
    plots.title('What We Get to See')

    sample.scatter('x', 'y', fit_line=True)
    plots.title('Regression Line: Estimate of True Line')

    sample.scatter('x', 'y', fit_line=True)
    plots.plot(xlims, true_slope*xlims + true_int, lw=2, color='green')
    plots.title("Regression Line and True Line")

## Linear regression

In [ ]:
def standard_units(x):
    """Converts an array x to standard units"""
    return (x - np.mean(x)) / np.std(x)

def correlation(t, x, y):
    x_su = standard_units(t.column(x))
    y_su = standard_units(t.column(y))
    return np.mean(x_su * y_su)

def slope(t, x, y):
    """t is a table; x and y are column labels"""
    r = correlation(t, x, y)
    slope_xy = r * np.std(t.column(y)) / np.std(t.column(x))
    return slope_xy

def intercept(t, x, y):
    """t is a table; x and y are column labels"""
    slope_xy = slope(t, x, y)
    intercept_xy = np.average(t.column(y) - slope_xy * np.average(t.column(x)))
    return intercept_xy

def fitted_values(t, x, y):
    """Return the height of the regression line at each x value"""
    a = slope(t, x, y)
    b = intercept(t, x, y)
    return a * t.column(x) + b

def residual(t, x, y):
    """Return the residuals of the regression line at each x value."""
    return t.column(y) - fitted_values(t, x, y)

def residual_plot(t, x, y):
    """displays a residual plot for the least squares regression line"""
    t = Table().with_columns(
            'fitted values', fitted_values(t, x, y),
            'residuals', residual(t, x, y)
    ).scatter('fitted values', 'residuals', color='r')
    plots.axhline(0, color='darkblue', lw=4)
    plots.title('Residual Plot')

In [ ]:
def add_regression_to_table(t, x, y):
    """ Returns a new table with 'Fitted value' and 'Residual' columns added"""
    t_with_extra_cols = t.with_columns(
        'Fitted value', fitted_values(t, x, y),
        'Residual', residual(t, x, y)
    )
    return t_with_extra_cols

## Regression Model ##

## Prediction ##

In [ ]:
births = Table.read_table('/home/jovyan/shared/Data/baby.csv')
births.show(3)

In [ ]:
births.scatter('Gestational Days', 'Birth Weight')

## Prediction - with Bootstrapping ##

In [ ]:
plots.figure(figsize=(6, 30))
plots.subplot(5, 1, 1)
plots.scatter(births[1], births[0], s=10)
plots.xlim([240, 310])
plots.title('Original sample')

for i in np.arange(1, 5, 1):
    plots.subplot(5,1,i+1)
    rep = births.sample()
    plots.scatter(rep[1], rep[0], s=10)
    plots.xlim([240, 310])
    plots.title('Bootstrap sample '+str(i))

In [ ]:
x = 280

for i in np.arange(4):
    resample = births.sample()
    predicted_y = prediction_at(resample, 'Gestational Days', 'Birth Weight', 280)
    print('Predicted y from bootstramp sample was', predicted_y)
    resample.scatter('Gestational Days', 'Birth Weight', fit_line=True)
    plots.scatter(280, predicted_y, color='gold', s=50, zorder=3);
    plots.plot([x, x], [40, predicted_y], color='gold', lw=2);
    plots.plot([240, x], [predicted_y, predicted_y], color='gold', lw=2);

## Regression Lines from Bootstrapping ##

In [ ]:
lines = Table(['slope','intercept', 'at 240', 'at 280', 'at 310'])

for i in range(10):
    resample = births.sample()
    a = slope(resample, 'Gestational Days', 'Birth Weight')
    b = intercept(resample, 'Gestational Days', 'Birth Weight')
    lines.append([a, b, a * 240 + b, a * 280 + b, a * 310 + b])

for i in np.arange(lines.num_rows):
    line = lines.row(i)
    plots.plot([240, 310], [line.item('at 240'), line.item('at 310')], lw=1)
    plots.scatter(280, line.item('at 280'), s=30, zorder=3)

In [ ]:
lines = Table(['slope','intercept', 'at 271', 'at 280', 'at 289'])

for i in range(10):
    resample = births.sample()
    a = slope(resample, 'Gestational Days', 'Birth Weight')
    b = intercept(resample, 'Gestational Days', 'Birth Weight')
    lines.append([a, b, a * 271 + b, a * 280 + b, a * 289 + b])

for i in np.arange(lines.num_rows):
    line = lines.row(i)
    plots.plot([271, 289], [line.item('at 271'), line.item('at 289')], lw=1)
    plots.scatter(280, line.item('at 280'), s=30, zorder=3)

## Confidence Interval for Prediction ##

In [ ]:
def bootstrap_prediction(t, x, y, new_x, repetitions=1000):

    # Bootstrap the scatter, predict, collect


    # Find the ends of the approximate 95% prediction interval


    # Display results
    Table().with_column('Prediction', predictions).hist(bins=20)
    plots.xlabel('predictions at x='+str(new_x))
    plots.plot([left, right], [0, 0], color='yellow', lw=8);
    print('Approximate 95%-confidence interval for height of true line:')
    print(left, right, '(width =', right - left, ')') 

## Confidence Interval for Prediction at x ##